In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generate a random dataset for classification
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a random forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the classifier on the training data
rf_classifier.fit(X_train, y_train)

# Make predictions on the test data
y_pred = rf_classifier.predict(X_test)

# Calculate the accuracy of the classifier
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.88


In [2]:
import numpy as np
from collections import Counter


class DecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth

    def fit(self, X, y):
        self.n_classes = len(set(y))
        self.n_features = X.shape[1]
        self.tree = self._grow_tree(X, y)

    def predict(self, X):
        return [self._predict(inputs) for inputs in X]

    def _grow_tree(self, X, y, depth=0):
        # Count the number of samples per class
        n_samples_per_class = [len(y[y == i]) for i in range(self.n_classes)]
        # Predict the class with the most samples as the majority class
        predicted_class = np.argmax(n_samples_per_class)

        # Create a node dictionary to store information about the node
        node = {
            'class': predicted_class,
            'samples': len(y),
            'value': n_samples_per_class,
            'depth': depth
        }

        if depth < self.max_depth:
            feature_indices = np.arange(self.n_features)
            best_gain = -1
            best_threshold = None
            best_indices = None

            for _ in range(self.n_features):
                # Randomly select a feature index
                random_feature = np.random.choice(feature_indices, size=1)
                # Find unique threshold values for the selected feature
                thresholds = np.unique(X[:, random_feature])

                for threshold in thresholds:
                    # Split the samples based on the selected feature and threshold
                    left_indices = np.where(X[:, random_feature] <= threshold)[0]
                    right_indices = np.where(X[:, random_feature] > threshold)[0]
                    # Calculate the information gain for the split
                    gain = self._information_gain(y, left_indices, right_indices)

                    if gain > best_gain:
                        best_gain = gain
                        best_threshold = threshold
                        best_indices = (left_indices, right_indices)

            if best_gain > 0:
                # Recursively grow the left and right branches of the tree
                left = self._grow_tree(X[best_indices[0], :], y[best_indices[0]], depth + 1)
                right = self._grow_tree(X[best_indices[1], :], y[best_indices[1]], depth + 1)
                node['feature_index'] = random_feature
                node['threshold'] = best_threshold
                node['left'] = left
                node['right'] = right

        return node

    def _predict(self, inputs):
        node = self.tree

        while 'threshold' in node:
            # Traverse the tree by following the selected feature and threshold
            if inputs[node['feature_index']] <= node['threshold']:
                node = node['left']
            else:
                node = node['right']

        return node['class']

    def _information_gain(self, y, left_indices, right_indices):
        p = len(left_indices) / len(y)
        q = len(right_indices) / len(y)
        # Calculate the information gain using entropy
        gain = self._entropy(y) - p * self._entropy(y[left_indices]) - q * self._entropy(y[right_indices])
        return gain

    def _entropy(self, y):
        counter = Counter(y)
        # Calculate the entropy of a set of labels
        probabilities = [count / len(y) for count in counter.values()]
        entropy = -np.sum(p * np.log2(p) for p in probabilities)
        return entropy


class RandomForest:
    def __init__(self, n_trees=100, max_depth=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        for _ in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth)
            # Randomly select samples with replacement
            indices = np.random.choice(len(X), size=len(X), replace=True)
            # Fit the decision tree on the selected samples
            tree.fit(X[indices], y[indices])
            self.trees.append(tree)

    def predict(self, X):
        # Make predictions by majority voting from all decision trees
        predictions = np.array([tree.predict(X) for tree in self.trees])
        majority_votes = np.apply_along_axis(lambda x: Counter(x).most_common(1)[0][0], axis=0, arr=predictions)
        return majority_votes


# Load the Iris dataset
def load_iris_dataset():
    from sklearn.datasets import load_iris

    iris = load_iris()
    X = iris.data
    y = iris.target
    return X, y


# Split the dataset into training and testing sets
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)

    indices = np.random.permutation(len(X))
    test_size = int(test_size * len(X))
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]
    X_train, X_test = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    return X_train, X_test, y_train, y_test


# Calculate the accuracy of predictions
def accuracy_score(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)


# Main code
X, y = load_iris_dataset()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a random forest classifier and fit the training data
random_forest = RandomForest(n_trees=100, max_depth=5)
random_forest.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = random_forest.predict(X_test)

# Calculate and print the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

/var/folders/g9/wk7rx71516lg31d3m_0wqlg80000gn/T/ipykernel_20427/2600680120.py:89: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  entropy = -np.sum(p * np.log2(p) for p in probabilities)


Accuracy: 0.8333333333333334


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a random forest classifier
random_forest = RandomForestClassifier(n_estimators=100, max_depth=5)

# Fit the classifier to the training data
random_forest.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = random_forest.predict(X_test)

# Calculate and print the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 1.0
